# A6 · Genes de la lista original de Laura que faltaban en el panel

**MVA Hackathon 2026 · Track 2** — Laura Vargas (con Claude Code), 14 oct 2026

**Pregunta:** la lista original del equipo tenía 201 genes, pero `config/genes_candidatos.txt` nunca los tuvo todos.
¿Cuáles faltaban y alguno de ellos podría explicar el MVA del paciente?

**Respuesta corta:** faltaban **31 genes** (más 4 símbolos que no existen en HGNC), entre ellos tres de los seis genes
de MVA con número en OMIM: ***CENATAC*** (MVA4), ***SLF2*** (MVA5) y ***SMC5*** (MVA6). Se analizaron con el mismo método del
diagnóstico diferencial (tareas A3 y A8, consecuencia en MANE Select): **ninguno tiene variantes raras funcionales con
patrón recesivo ni pérdida de función**. *BUB1B* sigue siendo la única explicación.

**De dónde sale la lista original:** `results/genes_candidatos_grch38.tsv`, generado por `scripts/resolve_genes.py` el
30 sep en la PC de Laura. Ese archivo está en `.gitignore` y además `resolve_genes.py` lo sobrescribe cada vez que corre,
por eso la celda 1 hace una copia y la lista quedó versionada en `config/lista_original_laura_201.tsv`.

**Cómo correrlo:** Colab conectado a un entorno de ejecución local (Jupyter iniciado en `C:\dev\mva-hackathon-2026`),
con el VCF en `data/raw/`. Los resultados se guardan en `data/a6/`, que no se versiona porque contiene variantes del paciente.

> **Reproducir en un clon nuevo:** cambiar `origen` en la celda 1 por `config/lista_original_laura_201.tsv`.
> Como los 31 genes ya se agregaron a `config/genes_candidatos.txt`, la celda 2 ahora daría *faltan: 0*; para repetir el
> análisis basta con escribir los 31 símbolos de la salida de la celda 3 en `data/a6/genes_a6.txt` y seguir desde la celda 4.

## Celda 1 · Copia de seguridad de la lista original
Copia la tabla de 201 genes a `data/a6/` (nunca pisa una copia anterior).

In [ ]:
import os, shutil
from pathlib import Path

os.chdir(r"C:\dev\mva-hackathon-2026")
A6 = Path("data/a6")
A6.mkdir(parents=True, exist_ok=True)

origen = Path("results/genes_candidatos_grch38.tsv")
copia  = A6 / "lista_original_laura_201.tsv"
if not copia.exists():                      # nunca pisa una copia anterior
    shutil.copy2(origen, copia)
print("Copia de seguridad:", copia, f"({copia.stat().st_size:,} bytes)")

## Celda 2 · ¿Qué genes de la lista original faltan en el panel?
Compara por el nombre escrito en la lista (`gen_lista`) **y** por el símbolo oficial (`simbolo`), para no contar como
faltante un gen renombrado (p. ej. ZNF828 → CHAMP1). Los 4 `no_encontrado` (MOXN1, RASRL1, TLX1-AS, UPP2-AS1) no
existen en HGNC y ya se habían cerrado en A6.

In [ ]:
import pandas as pd

mia = pd.read_csv(copia, sep="\t", dtype=str)

repo = set()
for linea in open("config/genes_candidatos.txt", encoding="utf-8"):
    linea = linea.split("#")[0].strip()      # quita comentarios
    if linea:
        repo.add(linea.split()[0])

falta = mia[~mia.gen_lista.isin(repo) & ~mia.simbolo.isin(repo)]
no_existen = falta[falta.estado == "no_encontrado"]
faltan     = falta[falta.estado != "no_encontrado"]

print(f"Tu lista: {len(mia)} · repo: {len(repo)} · faltan: {len(faltan)} (+{len(no_existen)} que no existen en HGNC)")
faltan[["gen_lista", "simbolo", "categoria", "tipo_gen"]]

## Celda 3 · Lista temporal con los 31 genes
Así se analizan solo estos genes, sin modificar el panel del equipo ni repetir el barrido de los 290.

In [ ]:
lista_a6 = A6 / "genes_a6.txt"
lista_a6.write_text("\n".join(faltan.simbolo) + "\n", encoding="utf-8")
print(lista_a6.read_text(encoding="utf-8"))

## Celda 4 · Diagnóstico diferencial sobre los 31 genes (3–10 min)
Usa `scripts/diagnostico_diferencial.py` **sin modificarlo**: desde fuera se cambia la lista de genes (`dd.GENES`) y la
carpeta de salida (`dd.OUT = data/a6/`). Método: variantes PASS → frecuencia en gnomAD v4 → las raras (AF < 1 %) que
parecen funcionales se reanotan con VEP en el transcrito MANE Select → patrón recesivo si hay ≥ 1 homocigota o
≥ 2 heterocigotas funcionales.

*Nota:* el mensaje final "Guardado en results/" está fijo en el script; los archivos quedaron en `data/a6/`.
Las 1.699 "ausentes de gnomAD" son sobre todo intrónicas: la lista de gnomAD por gen no las trae; VEP confirma que no
cambian la proteína.

In [ ]:
import sys, importlib.util

spec = importlib.util.spec_from_file_location("dd", "scripts/diagnostico_diferencial.py")
dd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dd)

dd.GENES = lista_a6                         # solo los 31 genes
dd.OUT   = A6                               # resultados en data/a6/
sys.argv = ["diagnostico_diferencial.py"]   # evita que Jupyter confunda al script
dd.main()

## Celda 5 · Resultado por gen y variantes sospechosas

In [ ]:
import os
import pandas as pd
from pathlib import Path

os.chdir(r"C:\dev\mva-hackathon-2026")
A6 = Path("data/a6")

# --- Resultado por gen (antes celda 5) ---
res = pd.read_csv(A6 / "diagnostico_diferencial_genes.tsv", sep="\t")
res["es_MVA"] = res.gen.isin(["CENATAC", "SMC5", "SLF2"])
cols = ["gen", "analizado", "motivo", "variantes_pass", "raras_funcionales",
        "descartadas_mane", "lof", "hom_funcionales", "het_funcionales", "patron_recesivo"]
print("Genes con patrón recesivo:", res.loc[res.patron_recesivo == True, "gen"].tolist() or "ninguno")
display(res.sort_values(["es_MVA", "raras_funcionales"], ascending=False)[cols])

# --- Detalle de variantes sospechosas (antes celda 6) ---
var = pd.read_csv(A6 / "diagnostico_diferencial_variantes.tsv", sep="\t")
print(f"\n{len(var)} variantes raras con efecto en algún transcrito; {var.cuenta.sum()} se mantienen en MANE")
# Sin posición ni lecturas: el repo será público y no se publican genotipos del paciente fuera de BUB1B.
# La tabla completa queda en data/a6/ (no versionado).
display(var[["gen", "zig", "consecuencia", "consecuencia_mane", "hgvsp", "cuenta", "af"]])

## Conclusión

| Comprobación | Resultado |
| --- | --- |
| Genes analizados | 30 de 31 (*IL40*, antes C17orf99, no está en gnomAD v4) |
| Patrón recesivo | **ninguno** |
| Pérdida de función (LOF) | **ninguna** |
| *CENATAC* / *SMC5* / *SLF2* | 11 / 87 / 67 variantes PASS, **0 raras funcionales** → descartados |
| Única alerta | *MINAR1* p.Phe405Leu, heterocigota, AF 0,49 % con 23 homocigotos en gnomAD → frecuente, sin relación con MVA |

**Ninguno de los 31 genes explica el MVA.** La conclusión de A3/A8 no cambia: *BUB1B* (p.Leu737Ter + p.Asn1002Lys) es
el único gen con dos variantes raras funcionales.

**Limitación:** 10 de los 31 son ARN no codificantes (9) o pseudogenes (1): HOTAIR, LINC00261, OIP5-AS1, etc. El método solo
detecta cambios en proteína, así que para ellos un 0 significa "no evaluable con este método", no "descartado".